# Analisando dados de produtos do pedido com Python

### Case - Preços de tabela

Quero investigar se os representantes usam muitos preços diferentes dos preços de tabela

In [47]:
# pip install pandas numpy openpyxl nbformat ipykernel plotly
# Passo 1: Importar a base de dados
import pandas as pd

#dados = pd.read_csv("OrderItem.csv")
dados = pd.read_csv("OrderItem.csv", encoding="cp1252")
#dados.columns.tolist()
#dados.columns[dados.columns.str.startswith("PRECO")].tolist()

# Passo 2: Visualizar a base de dados
# colunas inúteis - informações que não te ajudam, te atrapalham
# dados = dados.drop(columns="AUXCALCULO__C")
# display(dados)

# filtrar dados
from datetime import datetime
from dateutil.relativedelta import relativedelta  # pip install python-dateutil

ref = pd.Timestamp(datetime.now().date())  # ou pd.Timestamp("today").normalize()
corte = ref - relativedelta(months=6)
dados["DATA_NOTA_FISCAL__C"] = pd.to_datetime(dados["DATA_NOTA_FISCAL__C"], errors="coerce")
dados_recentes = dados[dados["DATA_NOTA_FISCAL__C"] >= corte]
#display(dados_recentes.tail(20))
display(dados_recentes[["UNITPRICE", "PRECO_2N__C", "PRECO_0__C", "PRECO_2__C", "PRECO_3__C", "PRECO_4__C", "PRECO_8__C"]].head(5))


C:\Users\User\AppData\Local\Temp\ipykernel_5704\210001056.py:6: DtypeWarning: Columns (0: ATACADISTA__C) have mixed types. Specify dtype option on import or set low_memory=False.
  dados = pd.read_csv("OrderItem.csv", encoding="cp1252")


,UNITPRICE,PRECO_2N__C,PRECO_0__C,PRECO_2__C,PRECO_3__C,PRECO_4__C,PRECO_8__C
415997,61.90,70.14,70.14,70.14,70.14,70.14,70.14
418627,63.00,70.14,70.14,70.14,70.14,70.14,70.14
420801,33.33,39.28,39.28,40.50,42.52,45.36,49.41
420802,33.33,39.28,39.28,40.50,42.52,45.36,49.41
433644,51.77,49.96,49.96,51.77,53.84,56.95,61.09


In [56]:
# Passo 3: Classificar preços
#display(dados_recentes.info())
#dados = dados.dropna()

import numpy as np

# nomes exatamente como no dados_recentes.columns
COL_UNIT = "UNITPRICE"
precos_ordem = ["PRECO_2__C", "PRECO_3__C", "PRECO_0__C", "PRECO_4__C", "PRECO_2N__C", "PRECO_8__C"]
atol = 0.005  # meio centavo de erro
up = dados_recentes[COL_UNIT].astype(float)
conds = []
labels = ["Tabela A", "Tabela B", "Tabela AA", "Tabela C", "Preco Min", "Preco Max", "Preco <Min", "Preco >Max"]

for col in precos_ordem:
    ref = dados_recentes[col].astype(float)
    conds.append(np.isclose(up, ref, rtol=0, atol=atol, equal_nan=False))
    #labels.append(col) - poderia fazer isso se usasse o nome da coluna como label

p2n = dados_recentes["PRECO_2N__C"].astype(float)
p8 = dados_recentes["PRECO_8__C"].astype(float)
conds.append(np.isfinite(p2n) & (up < p2n))
conds.append(np.isfinite(p8) & (up > p8))
dados_recentes = dados_recentes.copy()
dados_recentes["PrecoUsado"] = np.select(conds, labels, default="Outro preco")

print(f"{len(dados_recentes)} linhas em {dados_recentes.shape[1]} colunas")
display(dados_recentes["PrecoUsado"].value_counts())
display(dados_recentes["PrecoUsado"].value_counts(normalize=True).mul(100).round(2))

#-----------------------------------------------------------------------
#pegar intervalo onde está o preço de 'Outro preco'
dados_recentes_outros = dados_recentes[dados_recentes["PrecoUsado"]=="Outro preco"]
up = dados_recentes_outros["UNITPRICE"].astype(float)
p0 = dados_recentes_outros["PRECO_0__C"].astype(float)
p2 = dados_recentes_outros["PRECO_2__C"].astype(float)
p3 = dados_recentes_outros["PRECO_3__C"].astype(float)
p4 = dados_recentes_outros["PRECO_4__C"].astype(float)
p8 = dados_recentes_outros["PRECO_8__C"].astype(float)
labels2 = ["Entre min e AA", "Entre AA e A", "Entre A e B", "Entre B e C", "Entre C e Max"]
intervalos = [up < p0, (up > p0) & (up < p2), (up > p2) & (up < p3), (up > p3) & (up < p4), (up > p4) & (up < p8)]

#filtra só os que usaram outro preco
dados_recentes_outros["Intervalo"] = np.select(intervalos, labels2, default="Sem intervalo")
print(f"{len(dados_recentes_outros)} linhas em {dados_recentes_outros.shape[1]} colunas")
display(dados_recentes_outros["Intervalo"].value_counts())
display(dados_recentes_outros["Intervalo"].value_counts(normalize=True).mul(100).round(2))

111412 linhas em 194 colunas


PrecoUsado
Tabela A       82818
Tabela B       12165
Outro preco     9138
Preco <Min      3402
Tabela C        2674
Tabela AA        709
Preco >Max       312
Preco Max        194
Name: count, dtype: int64

PrecoUsado
Tabela A       74.33
Tabela B       10.92
Outro preco     8.20
Preco <Min      3.05
Tabela C        2.40
Tabela AA       0.64
Preco >Max      0.28
Preco Max       0.17
Name: proportion, dtype: float64

9138 linhas em 195 colunas


Intervalo
Entre A e B      2798
Entre B e C      2633
Entre AA e A     2431
Entre C e Max    1276
Name: count, dtype: int64

Intervalo
Entre A e B      30.62
Entre B e C      28.81
Entre AA e A     26.60
Entre C e Max    13.96
Name: proportion, dtype: float64

In [ ]:
# Passo 4:
